# BT-DKGRec-GCN — chạy thực nghiệm trên Google Colab

Đề án thạc sĩ: *Xây dựng ứng dụng dự báo hành vi khách hàng sử dụng đồ thị tri thức động*

| | |
|---|---|
| VPS | Mã nguồn, cấu trúc dự án, notebook, dựng KG cho demo, đẩy lên GitHub |
| **Colab (notebook này)** | **Toàn bộ thực nghiệm — mọi mô hình, mọi seed, mọi cohort** |

### Chống mất việc khi Colab ngắt giữa chừng

Colab có thể dừng đột ngột. Notebook này **không bao giờ phải làm lại từ đầu**:

| Giai đoạn | Lưu ở Drive | Khi chạy lại |
|---|---|---|
| Tiền xử lý (`data/interim`) | `cache/interim` | Nạp lại, bỏ qua nếu commit không đổi |
| Đồ thị (`data/processed`) | `cache/processed` | Nạp lại, bỏ qua nếu commit không đổi |
| Từng run | `runs/` — **đồng bộ ngay sau mỗi run** | Run đã xong thì bỏ qua |

Cache gắn với **commit** đang chạy. Đổi code → commit đổi → cache tự động bị bỏ, tính lại.
Không bao giờ có chuyện dùng nhầm dữ liệu cũ của phiên bản code khác.

⚠️ Không sửa siêu tham số trong notebook. Mọi tham số nằm trong `configs/`; sửa ở đó,
commit, `git pull` lại — nếu không, kết quả không tái lập được từ repo.


## 1. Môi trường

Ghi lại để đưa vào mục "môi trường thực nghiệm" của luận văn.

In [ ]:
import platform, subprocess, sys

print("Python :", sys.version.split()[0], "|", platform.platform())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                          "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
    print("GPU    :", gpu or "khong co")
except FileNotFoundError:
    print("GPU    : khong co nvidia-smi (dang chay CPU)")

with open("/proc/meminfo") as f:
    print(f"RAM    : {int(f.readline().split()[1]) / 1e6:.1f} GB")
print("\nMoc san (popularity) khong can GPU; cac mo hinh GCN o Buoc 6-8 thi can.")

## 2. Google Drive — dữ liệu vào, kết quả ra

Dữ liệu **tách hẳn khỏi repo**: repo chỉ có mã nguồn, dữ liệu nằm trên Drive.

```
MyDrive/BT-DKGRec/
├── raw/          4 file CSV RetailRocket   (bạn upload một lần)
├── cache/        interim + processed       (notebook tự tạo, để chạy lại nhanh)
└── runs/         run artifact              (notebook tự tạo, đồng bộ sau mỗi run)
```

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT  = Path("/content/drive/MyDrive/BT-DKGRec")
DRIVE_RAW   = DRIVE_ROOT / "raw"
DRIVE_CACHE = DRIVE_ROOT / "cache"
DRIVE_RUNS  = DRIVE_ROOT / "runs"
for d in (DRIVE_CACHE, DRIVE_RUNS):
    d.mkdir(parents=True, exist_ok=True)

print("Drive:", DRIVE_ROOT)
if DRIVE_RAW.exists():
    for f in sorted(DRIVE_RAW.iterdir()):
        print(f"  raw/{f.name:<30}{f.stat().st_size / 1e6:>9.1f} MB")
else:
    print("  CHUA CO raw/ — xem o buoc 5 de tai du lieu ve")
print(f"  runs/ dang co {len(list(DRIVE_RUNS.glob('*/metrics.json')))} run hoan chinh")

## 3. Mã nguồn

⚠️ **Điền `REPO_URL` trước khi chạy.** Repo rất nhẹ vì dữ liệu và kết quả đều bị `.gitignore` chặn.

Commit đang chạy được in ra và ghi kèm kết quả — đó là thứ gắn số liệu với mã nguồn.

In [ ]:
import os, subprocess
from pathlib import Path

REPO_URL = ""          # <-- DIEN URL GitHub cua repo vao day
BRANCH   = "main"
REPO_DIR = Path("/content/bt-dkgrec")

assert REPO_URL, "Chua dien REPO_URL"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)

COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
COMMIT_FULL = subprocess.run(["git", "log", "-1", "--format=%h %ad %s", "--date=short"],
                             capture_output=True, text=True).stdout.strip()
print("Commit:", COMMIT_FULL)

## 4. Thư viện

In [ ]:
!pip install -q -r requirements-train.txt

import importlib
for name in ("numpy", "pandas", "scipy", "pyarrow", "pydantic", "matplotlib", "torch"):
    try:
        m = importlib.import_module(name)
        print(f"  {name:<12}{getattr(m, '__version__', '?')}")
    except ImportError:
        print(f"  {name:<12}THIEU")

import torch
print("\nCUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 5. Dữ liệu thô

`data/raw` là **symlink** sang Drive — không copy 1,3 GB vào máy Colab.

Nếu Drive chưa có dữ liệu, bỏ comment khối Kaggle (cần `kaggle.json` đặt trong `MyDrive/BT-DKGRec/`).

In [ ]:
from pathlib import Path

RAW_FILES = ["events.csv", "item_properties_part1.csv",
             "item_properties_part2.csv", "category_tree.csv"]

# --- Neu Drive chua co du lieu, tai tu Kaggle ---
# !pip install -q kaggle
# !mkdir -p ~/.kaggle && cp "{DRIVE_ROOT}/kaggle.json" ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d retailrocket/ecommerce-dataset -p /tmp/rr --unzip
# !mkdir -p "{DRIVE_RAW}" && cp /tmp/rr/*.csv "{DRIVE_RAW}/"

raw = Path("data/raw")
raw.parent.mkdir(parents=True, exist_ok=True)
if raw.is_symlink() or raw.exists():
    raw.unlink()
raw.symlink_to(DRIVE_RAW)

missing = [f for f in RAW_FILES if not (raw / f).exists()]
assert not missing, f"Thieu file raw: {missing}"
for f in RAW_FILES:
    print(f"  OK  {f:<32}{(raw / f).stat().st_size / 1e6:>9.1f} MB")

## 6. Cơ chế checkpoint

Ba hàm nhỏ dùng cho toàn bộ notebook:

- `cache_valid(name)` — cache trên Drive có dùng lại được không (phải cùng commit)
- `cache_save(name, path)` / `cache_load(name, path)` — cất và nạp lại
- `sync_run(run_dir)` — đẩy một run lên Drive **ngay sau khi nó xong**

Cache gắn với commit, nên đổi code là cache tự hết hiệu lực.

In [ ]:
import shutil
from pathlib import Path

def _stamp(name: str) -> Path:
    return DRIVE_CACHE / f"{name}.commit"

def cache_valid(name: str) -> bool:
    """Cache dung duoc khi ton tai VA sinh ra tu dung commit dang chay."""
    stamp = _stamp(name)
    return (DRIVE_CACHE / name).exists() and stamp.exists() \
        and stamp.read_text().strip() == COMMIT

def cache_save(name: str, local: Path) -> None:
    target = DRIVE_CACHE / name
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(local, target)
    _stamp(name).write_text(COMMIT)
    size = sum(f.stat().st_size for f in target.rglob("*") if f.is_file())
    print(f"  cache -> Drive: {name} ({size / 1e6:.1f} MB)")

def cache_load(name: str, local: Path) -> None:
    if local.exists():
        shutil.rmtree(local)
    local.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(DRIVE_CACHE / name, local)
    print(f"  cache <- Drive: {name} (bo qua buoc tinh lai)")

def sync_run(run_dir: Path) -> None:
    """Day mot run len Drive ngay sau khi no xong."""
    shutil.copytree(run_dir, DRIVE_RUNS / run_dir.name, dirs_exist_ok=True)

def restore_runs() -> int:
    """Nap lai moi run da hoan chinh tu Drive ve may Colab."""
    local = Path("experiments/runs")
    local.mkdir(parents=True, exist_ok=True)
    n = 0
    for run in sorted(DRIVE_RUNS.glob("*/metrics.json")):
        target = local / run.parent.name
        if not target.exists():
            shutil.copytree(run.parent, target)
        n += 1
    return n

print(f"Da nap lai {restore_runs()} run tu Drive")

## 7. Kiểm tra khung dự án

**Test đỏ thì dừng lại.** Kết quả sinh ra từ code chưa xanh không có giá trị.

In [ ]:
!python scripts/00_check_setup.py
print("\n" + "=" * 70 + "\n")
!python -m pytest -q

## 8. Tiền xử lý — có checkpoint

Bảng audit đối chiếu với luận văn v11. Mốc cứng nhất: **train events = 2.024.042** (cohort
original). Lệch nhiều → sai file raw hoặc sai cách đọc, **dừng lại**.

Guard chống rò rỉ chạy tự động hai lượt mỗi lần preprocess (trong bộ nhớ, rồi đọc lại từ
Parquet). Không có cờ nào tắt được guard.

In [ ]:
%%time
from pathlib import Path

if cache_valid("interim"):
    cache_load("interim", Path("data/interim"))
else:
    for cohort in ("original", "active"):
        print(f"\n{'#' * 78}\n# tien xu ly cohort {cohort}\n{'#' * 78}")
        !python scripts/01_preprocess.py --cohort {cohort}
    cache_save("interim", Path("data/interim"))

## 9. Dựng đồ thị tri thức — có checkpoint

Ba biến thể dựng trong **cùng một lần chạy từ cùng một tập interim**, để không ai chất vấn
được rằng chúng đến từ dữ liệu khác nhau.

In [ ]:
%%time
from pathlib import Path

if cache_valid("processed"):
    cache_load("processed", Path("data/processed"))
else:
    for cohort in ("original", "active"):
        print(f"\n{'#' * 78}\n# dung graph cohort {cohort}\n{'#' * 78}")
        !python scripts/02_build_graph.py --cohort {cohort} --all
    cache_save("processed", Path("data/processed"))

In [ ]:
# Kiem chung cot loi: bt_dkgrec va static_kg_gcn phai giong het ve CAU TRUC,
# chi khac TRONG SO. Day la co so cua cau hoi hoi dong so 3 (DONG > TINH).
import json

import numpy as np
import scipy.sparse as sp

for cohort in ("original", "active"):
    a = sp.load_npz(f"data/processed/{cohort}/bt_dkgrec/adjacency.npz").tocsr()
    b = sp.load_npz(f"data/processed/{cohort}/static_kg_gcn/adjacency.npz").tocsr()
    sa = json.load(open(f"data/processed/{cohort}/bt_dkgrec/graph_stats.json"))
    sb = json.load(open(f"data/processed/{cohort}/static_kg_gcn/graph_stats.json"))
    same_pattern = np.array_equal(a.indptr, b.indptr) and np.array_equal(a.indices, b.indices)
    print(f"[{cohort}]  node {sa['n_nodes_total']:,}  canh {sa['n_edges_undirected']:,}")
    print(f"  cung sparsity pattern : {same_pattern}")
    print(f"  cung so canh tung loai: {sa['edges'] == sb['edges']}")
    print(f"  trong so KHAC nhau    : {not np.allclose(a.data, b.data)}")
    print(f"  khac dung mot thu     : {sa['weighting']} vs {sb['weighting']}\n")
    assert same_pattern and sa["edges"] == sb["edges"] and not np.allclose(a.data, b.data)

## 10. Chạy mô hình — có resume

Mọi mô hình chạy trên **cùng bộ seed** `[2020, 2021, 2022]`, không ngoại lệ.

- Run đã có trên Drive → **bỏ qua**
- Run vừa xong → **đẩy lên Drive ngay**, không đợi đến cuối

Nên nếu Colab ngắt, chạy lại ô này chỉ làm tiếp phần còn thiếu.

`popularity` và `recent_popularity` **tất định**: ba seed cho kết quả giống hệt nhau, độ
lệch chuẩn bằng 0. Đó là tính chất của mô hình, không phải lỗi — phải chú thích dưới bảng.

In [ ]:
%%time
import json, subprocess, time
from pathlib import Path

SEEDS      = json.load(open("experiments/seeds.json"))["seeds"]
MODELS     = ["popularity", "recent_popularity"]
COHORTS    = ["original", "active"]
EVAL_BATCH = 512      # Colab nhieu RAM; VPS chi chay duoc 16

def already_done(cohort, model, seed) -> bool:
    return any(p.parent.name.startswith(f"{cohort}_{model}_{seed}_")
               for p in DRIVE_RUNS.glob("*/metrics.json"))

runs_dir, failed, skipped = Path("experiments/runs"), [], 0
for cohort in COHORTS:
    for model in MODELS:
        for seed in SEEDS:
            tag = f"{cohort}/{model}/{seed}"
            if already_done(cohort, model, seed):
                print(f"  BO QUA {tag:<40}(da co tren Drive)")
                skipped += 1
                continue

            before = {p.name for p in runs_dir.iterdir() if p.is_dir()} if runs_dir.exists() else set()
            t0 = time.time()
            proc = subprocess.run(
                ["python", "scripts/03_train.py", "--model", model, "--cohort", cohort,
                 "--seed", str(seed), "--eval-batch-size", str(EVAL_BATCH)],
                capture_output=True, text=True,
            )
            if proc.returncode != 0:
                print(f"  FAIL   {tag:<40}{time.time() - t0:>6.0f}s")
                print(proc.stdout[-1500:], proc.stderr[-1500:])
                failed.append(tag)
                continue

            new = [p for p in runs_dir.iterdir() if p.is_dir() and p.name not in before]
            for run in new:
                sync_run(run)          # dong bo NGAY, khong doi den cuoi
            print(f"  OK     {tag:<40}{time.time() - t0:>6.0f}s  -> Drive")

total = len(COHORTS) * len(MODELS) * len(SEEDS)
print(f"\n{total - len(failed)}/{total} run san sang ({skipped} bo qua vi da co)")
assert not failed, f"Co run that bai: {failed}"

## 11. Kiểm tra run artifact

Mỗi run phải đủ 6 file. `curves.csv` bắt buộc — khi bảo vệ cần chứng minh baseline đã hội tụ chứ không bị dừng sớm.

In [ ]:
from pathlib import Path

REQUIRED = ["config.yaml", "seed.txt", "metrics.json", "topk.csv", "curves.csv", "train.log"]
runs = sorted(p for p in Path("experiments/runs").iterdir() if p.is_dir())

bad = 0
for run in runs:
    missing = [f for f in REQUIRED if not (run / f).exists()]
    if missing:
        bad += 1
        print(f"  THIEU {run.name:<52}{missing}")
print(f"{len(runs) - bad}/{len(runs)} run day du 6 file")
assert bad == 0

## 12. Bảng kết quả

Đọc **toàn bộ** run trong `experiments/runs/` — không có tham số lọc seed (quy tắc liêm chính 11).

Bảng in ra ở dạng Markdown, dán thẳng được vào luận văn.

In [ ]:
from pathlib import Path

import pandas as pd

from src.evaluation.reporting import format_table, load_runs, summarize

RUNS = Path("experiments/runs")
K = 20

warm = load_runs(RUNS, split="test", segment="warm")
cold = load_runs(RUNS, split="test", segment="cold")
valid_warm = load_runs(RUNS, split="valid", segment="warm")

print(f"BANG CHINH — test, phan doan warm, K={K}\n")
print(format_table(summarize(warm, k=K), k=K))

if not cold.empty:
    print(f"\n\nPHAN DOAN COLD — bao RIENG, khong tron vao bang chinh\n")
    print(format_table(summarize(cold, k=K), k=K))

In [ ]:
# Bang phu: K=10, va bang valid (dung de chon cau hinh, KHONG dung de bao cao ket qua cuoi)
print(f"K=10 — test, warm\n")
print(format_table(summarize(warm, k=10), k=10))
print(f"\n\nVALID — chi dung de chon cau hinh (quy tac 7)\n")
print(format_table(summarize(valid_warm, k=K), k=K))

## 13. Kiểm định thống kê

**Welch's t-test** (hai mẫu độc lập, phương sai không bằng nhau) — không dùng paired test,
vì phép toán thưa trên CUDA không cho kết quả bit-exact dù cố định seed.

Bốn cặp so sánh trả lời bốn câu hỏi hội đồng sẽ hỏi:

| # | Câu hỏi | Cặp so sánh |
|---|---|---|
| 1 | Có hơn cách ngây thơ không? | bt_dkgrec vs popularity |
| 2 | KG có ích không? | static_kg_gcn vs lightgcn |
| **3** | **ĐỘNG có hơn TĨNH không?** ★ | **bt_dkgrec vs static_kg_gcn** |
| 4 | Có cạnh tranh SOTA không? | bt_dkgrec vs lightgcn |

In [ ]:
from src.evaluation.reporting import compare_models

PAIRS = [
    ("bt_dkgrec", "popularity",     "1. Co hon cach ngay tho khong?"),
    ("static_kg_gcn", "lightgcn",   "2. KG co ich khong?"),
    ("bt_dkgrec", "static_kg_gcn",  "3. DONG co hon TINH khong?  <<< cot loi"),
    ("bt_dkgrec", "lightgcn",       "4. Co canh tranh SOTA khong?"),
]

available = set(warm["model"])
for cohort in sorted(warm["cohort"].unique()):
    print(f"\n{'=' * 78}\ncohort {cohort}\n{'=' * 78}")
    for a, b, question in PAIRS:
        if not {a, b} <= available:
            print(f"  {question}\n     (chua chay {sorted({a, b} - available)})")
            continue
        r = compare_models(warm, a, b, cohort, metric=f"ndcg@{K}")
        line = (f"  {question}\n     {a} {r['mean_a']:.6f} vs {b} {r['mean_b']:.6f}"
                f"  |  chenh {r['difference']:+.6f}")
        if r["p_value"] is not None:
            line += f"  |  p = {r['p_value']:.4f}"
            line += "  (co y nghia thong ke)" if r["p_value"] < 0.05 else "  (chua co y nghia)"
        else:
            line += f"  |  {r.get('note', '')}"
        print(line)

## 14. Biểu đồ

Hình được vẽ theo chuẩn dùng cho bài báo:

- **Màu gắn với mô hình, không gắn với thứ hạng** — mỗi mô hình một màu cố định trong mọi hình
- Bảng màu đã qua kiểm định mù màu (CVD ΔE 9.2 ≥ 8; normal-vision ΔE 27.6 ≥ 15)
- **Mỗi cột còn có hoa văn riêng và nhãn số in trực tiếp** — luận văn in đen trắng vẫn đọc được
- Không bao giờ dùng hai trục y
- Xuất cả **PNG 300 dpi** (chèn vào Word) và **PDF vector** (chèn vào LaTeX)

In [ ]:
from pathlib import Path

from IPython.display import Image, display

from src.evaluation.figures import (
    plot_ablation_pair,
    plot_model_comparison,
    plot_training_curves,
    plot_warm_cold,
)

FIG_DIR = Path("experiments/figures")

for cohort in sorted(warm["cohort"].unique()):
    png = plot_model_comparison(warm, cohort, FIG_DIR, k=K)
    display(Image(str(png)))

In [ ]:
# Hinh cot loi: DONG vs TINH — chi ve khi ca hai mo hinh da chay (Buoc 6-7)
if {"bt_dkgrec", "static_kg_gcn"} <= set(warm["model"]):
    display(Image(str(plot_ablation_pair(warm, FIG_DIR, k=K))))
else:
    print("Chua co bt_dkgrec / static_kg_gcn — hinh nay se ve duoc sau Buoc 6-7")

In [ ]:
# Warm vs cold — bao rieng, khong tron chung
for cohort in sorted(warm["cohort"].unique()):
    png = plot_warm_cold(warm, cold, cohort, FIG_DIR, k=K)
    if png:
        display(Image(str(png)))

# Duong hoi tu — bang chung baseline khong bi dung som (co tu Buoc 6 tro di)
for cohort in sorted(warm["cohort"].unique()):
    png = plot_training_curves(Path("experiments/runs"), cohort, FIG_DIR)
    if png:
        display(Image(str(png)))

## 15. Sao lưu lần cuối

Run artifact đã được đồng bộ sau mỗi lần chạy, nên ô này chỉ bổ sung **biểu đồ và bảng**.

Chạy trước khi đóng notebook.

In [ ]:
import shutil, time
from pathlib import Path

stamp = time.strftime("%Y%m%d-%H%M%S")
target = DRIVE_ROOT / "reports" / f"{stamp}_{COMMIT}"
target.mkdir(parents=True, exist_ok=True)

if FIG_DIR.exists():
    shutil.copytree(FIG_DIR, target / "figures", dirs_exist_ok=True)

(target / "table_test_warm.md").write_text(format_table(summarize(warm, k=K), k=K), encoding="utf-8")
if not cold.empty:
    (target / "table_test_cold.md").write_text(format_table(summarize(cold, k=K), k=K), encoding="utf-8")
(target / "COMMIT.txt").write_text(COMMIT_FULL + "\n", encoding="utf-8")

n_runs = len(list(DRIVE_RUNS.glob("*/metrics.json")))
print(f"Bao cao  -> {target}")
print(f"Run      -> {DRIVE_RUNS} ({n_runs} run)")
print(f"Commit   -> {COMMIT_FULL}")
print("\nDa sao luu day du. Dong notebook duoc roi.")